https://youtu.be/B55a_YoedgE?si=0T5M4iqesNpufvqS

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType
from pyspark.sql import functions as F

transaction_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("transaction_amount", FloatType(), True)
])

transactions_data = [
    (1, "credit", 30.0),
    (1, "debit", 90.0),
    (2, "credit", 50.0),
    (3, "debit", 57.0),
    (2, "debit", 90.0)
]

transactions_df = spark.createDataFrame(transactions_data, schema=transaction_schema)

transactions_df.display()

amount_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("current_amount", FloatType(), True)
])

amounts_data = [
    (1, 1000.0),
    (2, 2000.0),
    (3, 3000.0),
    (4, 4000.0)
]

amounts_df = spark.createDataFrame(amounts_data, schema=amount_schema)

amounts_df.display()

customer_id,transaction_type,transaction_amount
1,credit,30.0
1,debit,90.0
2,credit,50.0
3,debit,57.0
2,debit,90.0


customer_id,current_amount
1,1000.0
2,2000.0
3,3000.0
4,4000.0


In [0]:
(
    transactions_df
    .withColumn("transaction_amount",
                F.when(F.col("transaction_type")=="debit", F.col("transaction_amount")*-1).otherwise(F.col("transaction_amount")))
    .groupBy("customer_id")
    .agg(F.sum("transaction_amount").alias("transaction_amount"))
    .join(amounts_df, "customer_id", "full")
    .withColumn("transaction_amount",F.when(F.col("transaction_amount").isNull(),0).otherwise(F.col("transaction_amount")))
    .withColumn("current_amount",F.when(F.col("current_amount").isNull(),0).otherwise(F.col("current_amount")))
    .withColumn("balance",F.col("current_amount")+F.col("transaction_amount"))
    .select("customer_id","balance")
    .display()
)


customer_id,balance
1,940.0
2,1960.0
3,2943.0
4,4000.0
